# 04 — Embeddings, Indexation FAISS & Vérifications

**Objectif** : Vectoriser les child chunks via `mistral-embed`, les indexer dans FAISS, vérifier l'intégrité de l'index, et tester la qualité du retrieval.

**Entrées** :
- `data/interim/paris_chunks.parquet` (child chunks avec en-tête titre + mots-clés)
- `data/interim/paris_parent_docs.parquet` (parent documents)

**Sorties** :
- `models/faiss_index/` (index FAISS + docstore + parent_docs.json)

---

In [6]:
import json

from langchain_community.vectorstores import FAISS
import numpy as np
import pandas as pd

from notebooks.config import DATA_INTERIM, FAISS_INDEX_PATH, MISTRAL_API_KEY, MISTRAL_EMBED_MODEL
from notebooks.mistral import MistralEmbeddings

chunks_df = pd.read_parquet(DATA_INTERIM / "paris_chunks.parquet")
parent_df = pd.read_parquet(DATA_INTERIM / "paris_parent_docs.parquet")

embedder = MistralEmbeddings(api_key=MISTRAL_API_KEY, model=MISTRAL_EMBED_MODEL)

print(f"Child chunks       : {len(chunks_df)}")
print(f"Parent documents   : {len(parent_df)}")
print(f"Modèle d'embedding : {MISTRAL_EMBED_MODEL}")
print(f"Dimension attendue : {embedder.DIMENSION}")

Child chunks       : 350
Parent documents   : 200
Modèle d'embedding : mistral-embed
Dimension attendue : 1024


## 1. Création de l'index FAISS

In [7]:
print("Génération des embeddings via mistral-embed...")
texts = chunks_df["text"].tolist()
embeddings = embedder.embed_documents(texts)
print(f"Embeddings générés : {len(embeddings)} × {len(embeddings[0])} dimensions")

metadatas = [{"uid": str(row["uid"]), "chunk_id": str(row["chunk_id"])} for _, row in chunks_df.iterrows()]
text_embedding_pairs = list(zip(texts, embeddings))
vectorstore = FAISS.from_embeddings(
    text_embeddings=text_embedding_pairs,
    embedding=embedder,
    metadatas=metadatas,
)
print(f"Index FAISS construit : {vectorstore.index.ntotal} vecteurs")

index_path = str(FAISS_INDEX_PATH)
FAISS_INDEX_PATH.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(index_path)

parent_data = dict(zip(parent_df["uid"].astype(str), parent_df["parent_doc"]))
(FAISS_INDEX_PATH / "parent_docs.json").write_text(
    json.dumps(parent_data, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Index sauvegardé → {index_path}")

Génération des embeddings via mistral-embed...
Embeddings générés : 350 × 1024 dimensions
Index FAISS construit : 350 vecteurs
Index sauvegardé → /Users/kilian/Documents/Repositories/OpenClassrooms/OC-Concevez-et-deployez-un-systeme-RAG/models/faiss_index


## 2. Vérification de l'intégrité de l'index

On vérifie que :
- Tous les vecteurs sont de la bonne dimension
- Aucun vecteur n'est nul (embedding vide = chunk vide ou erreur)
- Le nombre de vecteurs correspond au nombre de chunks

In [8]:
n_vectors = vectorstore.index.ntotal
dim = len(embeddings[0])
all_vectors = np.array(embeddings)

norms = np.linalg.norm(all_vectors, axis=1)

print(f"Nombre de vecteurs : {n_vectors}")
print(f"Dimension          : {dim}")
print(f"Vecteurs nuls      : {(norms == 0).sum()}")
print(f"Norme min/max/moy  : {norms.min():.4f} / {norms.max():.4f} / {norms.mean():.4f}")

assert n_vectors == len(chunks_df), f"Mismatch: {n_vectors} vecteurs vs {len(chunks_df)} chunks"
assert (norms == 0).sum() == 0, "Vecteurs nuls détectés"
print("\nToutes les vérifications passent")

Nombre de vecteurs : 350
Dimension          : 1024
Vecteurs nuls      : 0
Norme min/max/moy  : 0.9999 / 1.0002 / 1.0000

Toutes les vérifications passent


## 3. Sauvegarde de l'index et du parent store

In [9]:
# Vérification du rechargement
vectorstore2 = FAISS.load_local(index_path, embedder, allow_dangerous_deserialization=True)
assert vectorstore2.index.ntotal == n_vectors
print(f"Rechargement vérifié : {vectorstore2.index.ntotal} vecteurs")

Rechargement vérifié : 350 vecteurs


## 4. Test du retrieval — 3 requêtes métier

On teste le système de bout en bout : requête → child chunk retrouvé → parent document associé.

Cela valide à la fois la qualité sémantique du retrieval et le mapping child → parent.

In [10]:
TEST_QUERIES = [
    "Concerts de jazz à Paris",
    "Activités gratuites en plein air",
    "Expositions d'art contemporain",
]

for query in TEST_QUERIES:
    print(f"\nREQUÊTE : « {query} »\n")
    results = vectorstore.similarity_search(query, k=5)
    for i, doc in enumerate(results, 1):
        uid = doc.metadata.get("uid")
        parent = parent_data.get(uid, doc.page_content)
        print(f"  {i}. [{uid}] {parent[:120]}...")
    print()


REQUÊTE : « Concerts de jazz à Paris »

  1. [49690984] # Jérémy Bruger Trio

**Date** : Le 26 mars 2026 à 18:30
**Lieu** : JASS CLUB, Paris
**Adresse** : 141 Rue de Tolbiac, P...
  2. [11643483] # Zoot Sundays! Sessions Jazz du dimanche

**Date** : Le 7 décembre 2025 à 18:30
**Lieu** : JASS CLUB, Paris
**Adresse**...
  3. [61473832] # Alexandre Herer 'Oxyd'

**Date** : Le 13 mars 2026 à 18:30
**Lieu** : JASS CLUB, Paris
**Adresse** : 141 Rue de Tolbia...
  4. [9469400] # Soirée jazz manouche & swing / par Alba Obert

**Date** : Le 6 juin 2026 à 17:30
**Lieu** : JASS CLUB, Paris
**Adresse...
  5. [90159916] # Jana Gavacova & Friends

**Date** : Le 3 décembre 2025 à 18:30
**Lieu** : JASS CLUB, Paris
**Adresse** : 141 Rue de To...


REQUÊTE : « Activités gratuites en plein air »

  1. [89411008] # Action collective

**Date** : Le 28 mai 2026 à 09:00
**Lieu** : Association Dessine-moi un mouton, Paris
**Adresse** :...
  2. [78571514] # Circuits "Passages et Galeries"

**Date** : Du 19 se

## Bilan

| Vérification | Résultat |
|---|---|
| Modèle d'embedding | `mistral-embed` (1024 dimensions) |
| Vecteurs nuls | Aucun |
| Nombre de vecteurs vs chunks | Identique |
| Sauvegarde / rechargement | OK |
| Round-trip child → parent | Fonctionnel via `FAISSStore.search()` |

---
**Suite** : `05.0-kl-rag-chain.ipynb` — intégration chaîne RAG + Mistral